<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Wan-VACE_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/Wan-VACE_Colab.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-View_Source-181717?logo=github" alt="View on GitHub"/></a>

# 🎬 Wan 2.1 VACE — All-in-One Video Creation & Editing (`1.3B` & `14B` Diffusers)

**Models:** [`Wan-AI/Wan2.1-VACE-1.3B-diffusers`](https://huggingface.co/Wan-AI/Wan2.1-VACE-1.3B-diffusers) &nbsp;|&nbsp; [`Wan-AI/Wan2.1-VACE-14B-diffusers`](https://huggingface.co/Wan-AI/Wan2.1-VACE-14B-diffusers) &nbsp;|&nbsp; **Paper & GitHub:** [`ali-vilab/VACE`](https://github.com/ali-vilab/VACE) &nbsp;|&nbsp; **License:** Apache 2.0

**Wan 2.1 VACE** (*Video All-in-one Creation and Editing*) unifies **Reference-to-Video (R2V)**, **Text-to-Video (T2V)**, **Video-to-Video (V2V) Restyling**, **Masked Video Inpainting (Swap / Move-Anything)**, **First-Last-Frame Interpolation (`FLF2V`)**, and **Spatial Video Outpainting** inside a single `WanVACEPipeline` architecture.

### ✨ Key Features of This Colab Notebook
- **Dual Official Diffusers Models**:
  - **`1.3B-VACE (Fast, T4/L4 Default)`** (`Wan-AI/Wan2.1-VACE-1.3B-diffusers`) — requires only **~8.5–11 GB VRAM** with block-level group offload, running smoothly on free **Colab T4 (16 GB)** and **L4 (24 GB)** GPUs.
  - **`14B-VACE (Flagship, L4/A100)`** (`Wan-AI/Wan2.1-VACE-14B-diffusers`) — 14B flagship model for maximum temporal fidelity and 720p generation; fits **L4 (24 GB)** and **A100 (40/80 GB)** using `torchao` INT8 weight-only quantization + Diffusers `enable_group_offload(offload_type="block_level", use_stream=True)`.
- **3 Dedicated Workflow Tabs (4 Core VACE Tasks)**:
  1. **🖼️ Reference-to-Video (R2V) & Text-to-Video**: Upload 1–3 subject/character/object reference images (`reference_images`) + text prompt to synthesize a new video featuring those exact subjects, or leave reference images empty for pure T2V.
  2. **🎭 Video-to-Video (V2V) & Masked Inpainting (Swap / Move-Anything)**: Upload a source video + optional mask video/image (or interactive bounding-box region `x1, y1, x2, y2`) + optional reference image to swap characters, replace objects, or restyle scenes.
  3. **🎞️ First/Last Keyframe (`FLF2V`) & Outpainting Conditioning**: Automatically builds the `(128, 128, 128)` neutral-gray conditioning video and `L`-mode binary mask (`0` = preserve, `255` = generate) from uploaded start/end keyframes or top/bottom/left/right outpainting margins.
- **Persistent Google Drive Cache & Outputs**:
  - Model weights are cached under `/content/drive/MyDrive/AEI_3D_Cache/Wan-VACE` and rendered `.mp4` videos are saved to `/content/drive/MyDrive/AEI_3D_Out/Wan-VACE` (ready for 1-click 1080p/4K upscaling in `SeedVR2_Colab.ipynb`).

| GPU Tier | VRAM | Recommended Variant | Quantization & Offload | Typical Resolution |
| :--- | :--- | :--- | :--- | :--- |
| **Colab T4** | 16 GB | `1.3B-VACE (Fast, T4/L4)` | BF16/FP16 + Block-Level Group Offload | `832 × 480` (480p, 49–81 frames) |
| **Colab L4** | 24 GB | `1.3B-VACE` or `14B-VACE` | `14B`: `torchao` INT8 + Group Offload | `832 × 480` or `1280 × 720` (720p) |
| **Colab A100** | 40 / 80 GB | `14B-VACE (Flagship, L4/A100)` | BF16 or INT8 + Group Offload | `1280 × 720` (720p, 81 frames) |

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs `diffusers` (with `WanVACEPipeline` & `AutoencoderKLWan`), `transformers`, `accelerate`,
# @markdown `torchao` (for 14B INT8 quantization), `opencv-python-headless`, `imageio[ffmpeg]`, `tqdm==4.66.5`, and `gradio==5.49.1`.

import os, sys, subprocess, time, torch

if not torch.cuda.is_available():
    raise SystemExit('No CUDA GPU detected. Go to Runtime -> Change runtime type -> select T4, L4, or A100 GPU.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'[GPU] {gpu_name} — {vram_gb:.1f} GB VRAM | PyTorch {torch.__version__} (CUDA {torch.version.cuda})')

t0 = time.time()
print('[pip] Installing diffusers (WanVACEPipeline) + torchao + media packages + gradio==5.49.1 ...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/huggingface/diffusers',
    'transformers>=4.49.0',
    'accelerate>=1.1.0',
    'huggingface-hub>=0.27.0',
    'peft>=0.14.0',
    'torchao',
    'imageio[ffmpeg]',
    'imageio-ffmpeg',
    'av',
    'opencv-python-headless',
    'sentencepiece',
    'ftfy',
    'regex',
    'tqdm==4.66.5',
    'gradio==5.49.1',
], check=True)
print(f'[Install] Completed in {time.time() - t0:.1f}s.')

try:
    from diffusers import WanVACEPipeline, AutoencoderKLWan, UniPCMultistepScheduler
    from diffusers.utils import export_to_video, load_image, load_video
    print('[Install] Verified: WanVACEPipeline, AutoencoderKLWan, and UniPCMultistepScheduler imported cleanly.')
except Exception as exc:
    raise SystemExit(f'Failed to import WanVACEPipeline from diffusers: {exc}')

In [ ]:
# @title STEP 2 — Mount Google Drive & Pre-Cache Wan-VACE Weights
# @markdown Mounts Google Drive and caches model weights under `/content/drive/MyDrive/AEI_3D_Cache/Wan-VACE`
# @markdown so subsequent Colab sessions start without re-downloading.

import os, sys, time, pathlib, shutil
from huggingface_hub import snapshot_download
from tqdm.notebook import tqdm

MODEL_REGISTRY = {
    '1.3B-VACE (Fast, T4/L4)': (
        'Wan-AI/Wan2.1-VACE-1.3B-diffusers',
        16.5,
        12,
        '1.3B all-in-one VACE model (R2V, V2V, Inpainting, FLF2V, Outpainting) — fits T4 16GB & L4 24GB',
    ),
    '14B-VACE (Flagship, L4/A100)': (
        'Wan-AI/Wan2.1-VACE-14B-diffusers',
        44.0,
        24,
        '14B flagship VACE model — fits L4 24GB & A100 40GB with torchao INT8 + block-level group offload',
    ),
}

DEFAULT_VARIANT = '1.3B-VACE (Fast, T4/L4)'  # @param ['1.3B-VACE (Fast, T4/L4)', '14B-VACE (Flagship, L4/A100)']
PRECACHE_DEFAULT_ON_RUN = True  # @param {type:"boolean"}

try:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/Wan-VACE')
    HAS_DRIVE = True
except Exception as exc:
    print(f'[Cache] Google Drive unavailable ({exc}); using local /content/hf_cache/Wan-VACE.')
    CACHE_ROOT = pathlib.Path('/content/hf_cache/Wan-VACE')
    HAS_DRIVE = False

HF_HOME_DIR = CACHE_ROOT / 'hf_home'
HF_HUB_DIR = HF_HOME_DIR / 'hub'
HF_HUB_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME_DIR)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_HUB_DIR)
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

OUT_DIR = pathlib.Path('/content/wan_vace_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)
if HAS_DRIVE:
    DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/Wan-VACE')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
else:
    DRIVE_OUT = OUT_DIR

def _dir_size_bytes(path: pathlib.Path) -> int:
    if not path.exists():
        return 0
    return sum(f.stat().st_size for f in path.rglob('*') if f.is_file())

def download_vace_variant(variant_key: str, force: bool = False) -> str:
    if variant_key not in MODEL_REGISTRY:
        raise ValueError(f'Unknown variant {variant_key}. Valid choices: {list(MODEL_REGISTRY)}')
    repo_id, disk_gb, _, blurb = MODEL_REGISTRY[variant_key]
    local_dir = CACHE_ROOT / repo_id.replace('/', '_')
    existing_gb = _dir_size_bytes(local_dir) / (1024 ** 3)
    if existing_gb >= disk_gb * 0.75 and (local_dir / 'model_index.json').exists() and not force:
        print(f'[Cache] {variant_key} already cached at {local_dir} ({existing_gb:.1f} GB).')
        return str(local_dir)
    print(f'[Cache] Downloading {variant_key} ({repo_id}, ~{disk_gb:.1f} GB) — {blurb} ...')
    t0 = time.time()
    snapshot_download(
        repo_id=repo_id,
        local_dir=str(local_dir),
        local_dir_use_symlinks=False,
        ignore_patterns=['assets/*', '*.md'],
        tqdm_class=tqdm,
    )
    print(f'[Cache] Download complete in {time.time() - t0:.1f}s -> {local_dir}')
    return str(local_dir)

print(f"[Cache] HF_HOME    : {os.environ['HF_HOME']}")
print(f'[Cache] Output dir : {DRIVE_OUT}')
if PRECACHE_DEFAULT_ON_RUN:
    DEFAULT_CKPT_DIR = download_vace_variant(DEFAULT_VARIANT)
    print(f'[Setup] Default checkpoint ready at: {DEFAULT_CKPT_DIR}')
else:
    print('[Setup] Pre-download skipped; weights will download automatically on first generation.')

In [ ]:
# @title STEP 3 — Wan-VACE Core Pipeline & Conditioning Builders
# @markdown Defines `WanVACEEngine` and conditioning builders for all 4 VACE workflows:
# @markdown 1. **Reference-to-Video (R2V) & Text-to-Video (T2V)** (`reference_images` + `prompt`)
# @markdown 2. **Video-to-Video (V2V) & Masked Video Inpainting** (`video` + `mask` + optional `reference_images`)
# @markdown 3. **First/Last Keyframe (`FLF2V`) Conditioning** (`first_frame` + optional `last_frame`)
# @markdown 4. **Spatial Video/Image Outpainting** (`top`, `bottom`, `left`, `right` padding margins)

import os, sys, time, gc, uuid, random, shutil, pathlib, traceback
from typing import List, Optional, Tuple
import cv2, torch, numpy as np
from PIL import Image, ImageOps
from IPython.display import display, FileLink
from diffusers import WanVACEPipeline, AutoencoderKLWan, UniPCMultistepScheduler
from diffusers.utils import export_to_video, load_image

DEFAULT_FPS = 16
MOD_VALUE = 16  # WanVACE spatial dimensions must be divisible by vae_scale_factor_spatial (8) * patch_size (2) = 16
NEGATIVE_PROMPT_DEFAULT = (
    'Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, '
    'static, overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, '
    'extra fingers, poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, '
    'fused fingers, still picture, messy background, three legs, many people in the background, walking backwards'
)

def snap_hw(val: int, mod: int = MOD_VALUE, lo: int = 256, hi: int = 1280) -> int:
    v = max(lo, min(hi, int(val)))
    return max(mod, (v // mod) * mod)

def snap_frames(num_frames: int) -> int:
    """Wan 2.1 requires (num_frames - 1) to be divisible by 4."""
    n = max(5, int(num_frames))
    return ((n - 1) // 4) * 4 + 1

def to_rgb_pil(img_input, target_size: Optional[Tuple[int, int]] = None) -> Image.Image:
    if isinstance(img_input, (str, pathlib.Path)):
        pil = Image.open(str(img_input)).convert('RGB')
    elif isinstance(img_input, np.ndarray):
        pil = Image.fromarray(img_input.astype(np.uint8)).convert('RGB')
    elif isinstance(img_input, Image.Image):
        pil = img_input.convert('RGB')
    else:
        raise ValueError(f'Unsupported image input type: {type(img_input)}')
    if target_size is not None:
        pil = pil.resize(target_size, Image.Resampling.LANCZOS)
    return pil

def read_video_frames_pil(video_path: str, width: int, height: int, max_frames: int = 81) -> Tuple[List[Image.Image], float]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f'Could not open video file: {video_path}')
    fps = cap.get(cv2.CAP_PROP_FPS) or DEFAULT_FPS
    frames = []
    while len(frames) < max_frames:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        pil = Image.fromarray(frame_rgb).resize((width, height), Image.Resampling.LANCZOS)
        frames.append(pil)
    cap.release()
    if not frames:
        raise ValueError(f'No readable frames found in {video_path}')
    target_n = snap_frames(len(frames))
    while len(frames) < target_n:
        frames.append(frames[-1].copy())
    return frames[:target_n], float(fps)

def prepare_flf2v_video_and_mask(
    first_img: Image.Image,
    last_img: Optional[Image.Image],
    height: int,
    width: int,
    num_frames: int,
) -> Tuple[List[Image.Image], List[Image.Image]]:
    """Constructs video + mask frames for First-Frame (I2V) or First-Last-Frame (FLF2V) conditioning."""
    num_frames = snap_frames(num_frames)
    first_pil = to_rgb_pil(first_img, target_size=(width, height))
    gray_frame = Image.new('RGB', (width, height), (128, 128, 128))
    mask_keep = Image.new('L', (width, height), 0)
    mask_gen = Image.new('L', (width, height), 255)
    if last_img is not None:
        last_pil = to_rgb_pil(last_img, target_size=(width, height))
        frames = [first_pil] + [gray_frame.copy() for _ in range(num_frames - 2)] + [last_pil]
        mask = [mask_keep] + [mask_gen.copy() for _ in range(num_frames - 2)] + [mask_keep.copy()]
    else:
        frames = [first_pil] + [gray_frame.copy() for _ in range(num_frames - 1)]
        mask = [mask_keep] + [mask_gen.copy() for _ in range(num_frames - 1)]
    return frames, mask

def prepare_outpaint_video_and_mask(
    source_frames: List[Image.Image],
    height: int,
    width: int,
    num_frames: int,
    pad_top: int = 64,
    pad_bottom: int = 64,
    pad_left: int = 128,
    pad_right: int = 128,
) -> Tuple[List[Image.Image], List[Image.Image]]:
    """Constructs video + mask frames for spatial outpainting around an image or video."""
    num_frames = snap_frames(num_frames)
    pad_top = max(0, min(height // 3, int(pad_top)))
    pad_bottom = max(0, min(height // 3, int(pad_bottom)))
    pad_left = max(0, min(width // 3, int(pad_left)))
    pad_right = max(0, min(width // 3, int(pad_right)))
    inner_w = max(64, width - pad_left - pad_right)
    inner_h = max(64, height - pad_top - pad_bottom)

    mask_frame = Image.new('L', (width, height), 255)
    inner_keep = Image.new('L', (inner_w, inner_h), 0)
    mask_frame.paste(inner_keep, (pad_left, pad_top))

    frames_out, masks_out = [], []
    for i in range(num_frames):
        src_f = source_frames[min(i, len(source_frames) - 1)]
        inner_pil = to_rgb_pil(src_f, target_size=(inner_w, inner_h))
        canvas = Image.new('RGB', (width, height), (128, 128, 128))
        canvas.paste(inner_pil, (pad_left, pad_top))
        frames_out.append(canvas)
        masks_out.append(mask_frame.copy())
    return frames_out, masks_out

def prepare_v2v_inpaint_video_and_mask(
    source_video_path: str,
    height: int,
    width: int,
    num_frames: int,
    mask_mode: str = 'Bounding Box Region (Inpaint / Swap)',
    mask_media_path: Optional[str] = None,
    bbox_norm: Tuple[float, float, float, float] = (0.25, 0.15, 0.75, 0.85),
    invert_mask: bool = False,
) -> Tuple[List[Image.Image], List[Image.Image]]:
    """Constructs video + mask frames for V2V restyling or masked video object/character inpainting."""
    src_frames, _ = read_video_frames_pil(source_video_path, width=width, height=height, max_frames=snap_frames(num_frames))
    n = len(src_frames)
    masks = []
    if mask_mode.startswith('Full Video'):
        m = Image.new('L', (width, height), 255)
        masks = [m.copy() for _ in range(n)]
    elif mask_mode.startswith('Uploaded Mask') and mask_media_path and os.path.exists(str(mask_media_path)):
        ext = pathlib.Path(str(mask_media_path)).suffix.lower()
        if ext in ('.mp4', '.mov', '.avi', '.mkv', '.webm'):
            m_frames, _ = read_video_frames_pil(str(mask_media_path), width=width, height=height, max_frames=n)
            masks = [f.convert('L') for f in m_frames]
        else:
            m_single = Image.open(str(mask_media_path)).convert('L').resize((width, height), Image.Resampling.NEAREST)
            masks = [m_single.copy() for _ in range(n)]
    else:
        x1, y1, x2, y2 = bbox_norm
        px1 = int(round(max(0.0, min(0.95, float(x1))) * width))
        py1 = int(round(max(0.0, min(0.95, float(y1))) * height))
        px2 = int(round(max(float(x1) + 0.05, min(1.0, float(x2))) * width))
        py2 = int(round(max(float(y1) + 0.05, min(1.0, float(y2))) * height))
        m = Image.new('L', (width, height), 0)
        box_white = Image.new('L', (max(1, px2 - px1), max(1, py2 - py1)), 255)
        m.paste(box_white, (px1, py1))
        masks = [m.copy() for _ in range(n)]

    if invert_mask:
        masks = [ImageOps.invert(m) for m in masks]
    return src_frames, masks

class WanVACEEngine:
    """Lazy loader & inference engine for Wan2.1-VACE-1.3B-diffusers and Wan2.1-VACE-14B-diffusers."""
    def __init__(self):
        self.pipeline = None
        self.vae = None
        self.variant = None
        self.quantized = False
        self.offloaded = False
        self.dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

    def unload(self):
        if self.pipeline is not None:
            del self.pipeline
            self.pipeline = None
        if self.vae is not None:
            del self.vae
            self.vae = None
        self.variant = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f'[Engine] Unloaded pipeline. Allocated VRAM: {torch.cuda.memory_allocated() / (1024**3):.2f} GB')

    def load(
        self,
        variant: str = DEFAULT_VARIANT,
        offload: bool = True,
        quantize_int8: bool = False,
        force_reload: bool = False,
    ):
        if variant not in MODEL_REGISTRY:
            raise ValueError(f'Unknown variant: {variant}. Choose from {list(MODEL_REGISTRY)}')
        if torch.cuda.is_available():
            vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
            if variant.startswith('14B') and vram_gb < 32.0 and not quantize_int8:
                print(f'[Load] Auto-enabling torchao INT8 quantization for {variant} on {vram_gb:.1f} GB GPU.')
                quantize_int8 = True

        if (
            self.pipeline is not None
            and self.variant == variant
            and self.quantized == bool(quantize_int8)
            and self.offloaded == bool(offload)
            and not force_reload
        ):
            return

        if self.pipeline is not None:
            self.unload()

        local_dir = download_vace_variant(variant)
        print(f'[Load] Loading {variant} from {local_dir} (dtype={self.dtype}, INT8={quantize_int8}, offload={offload}) ...')
        t0 = time.time()
        self.vae = AutoencoderKLWan.from_pretrained(
            local_dir, subfolder='vae', torch_dtype=torch.float32
        )
        self.pipeline = WanVACEPipeline.from_pretrained(
            local_dir, vae=self.vae, torch_dtype=self.dtype
        )
        self.pipeline.scheduler = UniPCMultistepScheduler.from_config(
            self.pipeline.scheduler.config, flow_shift=3.0
        )

        if quantize_int8:
            try:
                from torchao.quantization import quantize_, Int8WeightOnlyConfig
                print('[Load] Quantizing WanVACETransformer3DModel to INT8 via torchao ...')
                quantize_(self.pipeline.transformer, Int8WeightOnlyConfig())
            except Exception as exc:
                print(f'[Load] torchao INT8 quantization skipped ({exc}); continuing in {self.dtype}.')

        if offload:
            grouped = False
            if hasattr(self.pipeline, 'enable_group_offload'):
                try:
                    self.pipeline.enable_group_offload(
                        onload_device=torch.device('cuda'),
                        offload_device=torch.device('cpu'),
                        offload_type='block_level',
                        num_blocks_per_group=1,
                        use_stream=True,
                    )
                    grouped = True
                    print('[Load] Enabled block-level group offload with CUDA streams.')
                except Exception as exc:
                    print(f'[Load] Group offload fallback ({exc}) -> enable_model_cpu_offload().')
            if not grouped:
                self.pipeline.enable_model_cpu_offload()
                print('[Load] Enabled standard model CPU offload.')
        else:
            self.pipeline.to('cuda')

        if hasattr(self.pipeline.vae, 'enable_tiling'):
            self.pipeline.vae.enable_tiling()
        if hasattr(self.pipeline.vae, 'enable_slicing'):
            self.pipeline.vae.enable_slicing()

        self.variant = variant
        self.quantized = bool(quantize_int8)
        self.offloaded = bool(offload)
        print(f'[Load] Ready in {time.time() - t0:.1f}s | Allocated VRAM: {torch.cuda.memory_allocated() / (1024**3):.2f} GB')

    def generate(
        self,
        prompt: str,
        negative_prompt: str = NEGATIVE_PROMPT_DEFAULT,
        video: Optional[List[Image.Image]] = None,
        mask: Optional[List[Image.Image]] = None,
        reference_images: Optional[List[Image.Image]] = None,
        conditioning_scale: float = 1.0,
        height: int = 480,
        width: int = 832,
        num_frames: int = 49,
        num_inference_steps: int = 30,
        guidance_scale: float = 5.0,
        flow_shift: float = 3.0,
        seed: int = 42,
        fps: int = DEFAULT_FPS,
    ) -> str:
        if self.pipeline is None:
            raise RuntimeError('WanVACEEngine is not loaded. Call ENGINE.load(...) first.')
        height = snap_hw(height)
        width = snap_hw(width)
        num_frames = snap_frames(num_frames)
        actual_seed = int(seed) if int(seed) >= 0 else random.randint(0, 2**31 - 1)
        generator = torch.Generator(device='cpu').manual_seed(actual_seed)

        self.pipeline.scheduler = UniPCMultistepScheduler.from_config(
            self.pipeline.scheduler.config, flow_shift=float(flow_shift)
        )

        clean_refs = None
        if reference_images:
            valid_refs = [to_rgb_pil(img) for img in reference_images if img is not None]
            if valid_refs:
                clean_refs = valid_refs

        with torch.inference_mode():
            out = self.pipeline(
                prompt=prompt,
                negative_prompt=negative_prompt or NEGATIVE_PROMPT_DEFAULT,
                video=video,
                mask=mask,
                reference_images=clean_refs,
                conditioning_scale=float(conditioning_scale),
                height=height,
                width=width,
                num_frames=num_frames,
                num_inference_steps=int(num_inference_steps),
                guidance_scale=float(guidance_scale),
                generator=generator,
            ).frames[0]

        out_path = OUT_DIR / f'wan_vace_{int(time.time())}_{uuid.uuid4().hex[:6]}.mp4'
        export_to_video(out, str(out_path), fps=int(fps))
        return str(out_path)

ENGINE = WanVACEEngine()
print('[Core] WanVACEEngine initialized. Supported variants:', list(MODEL_REGISTRY.keys()))

In [ ]:
# @title STEP 4 — Launch Gradio UI (R2V, V2V Inpainting, Keyframe FLF2V & Outpainting)
# @markdown Launches the interactive Gradio studio with dedicated tabs for **Reference-to-Video (R2V) / T2V**,
# @markdown **Video-to-Video (V2V) & Masked Inpainting**, **First/Last Keyframe (`FLF2V`) & Outpainting**, and **VRAM Management**.

import os, sys, time, uuid, shutil, pathlib, traceback
import torch, numpy as np, gradio as gr
from PIL import Image
from IPython.display import clear_output, display, FileLink

RES_PRESETS = [
    ('832 × 480 (16:9 Landscape, 480p — Recommended T4/L4)', 480, 832),
    ('480 × 832 (9:16 Portrait, 480p — Recommended T4/L4)', 832, 480),
    ('512 × 512 (1:1 Square, 480p)', 512, 512),
    ('1280 × 720 (16:9 Landscape, 720p — L4/A100)', 720, 1280),
    ('720 × 1280 (9:16 Portrait, 720p — L4/A100)', 1280, 720),
]
RES_NAMES = [r[0] for r in RES_PRESETS]

def _resolve_res(res_label: str):
    for label, h, w in RES_PRESETS:
        if label == res_label:
            return h, w
    return 480, 832

def _save_to_drive(local_mp4: str, prefix: str = 'vace') -> str:
    dest = DRIVE_OUT / f'{prefix}_{int(time.time())}_{uuid.uuid4().hex[:6]}.mp4'
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local_mp4, str(dest))
    return str(dest)

def ui_run_r2v(
    variant, quantize_int8, offload, prompt, neg_prompt,
    ref1, ref2, ref3, res_label, num_frames, steps, cfg, cond_scale, shift, seed,
    progress=gr.Progress(),
):
    if not prompt or not prompt.strip():
        return None, '❌ Please enter a text prompt.'
    try:
        progress(0.05, desc=f'Loading {variant} ...')
        ENGINE.load(variant=variant, offload=bool(offload), quantize_int8=bool(quantize_int8))
        h, w = _resolve_res(res_label)
        refs = [r for r in (ref1, ref2, ref3) if r is not None]
        mode_str = f'R2V ({len(refs)} ref image(s))' if refs else 'Pure Text-to-Video (T2V)'
        progress(0.15, desc=f'Running {mode_str} at {w}x{h} ({int(num_frames)} frames, {int(steps)} steps) ...')
        t0 = time.time()
        out_mp4 = ENGINE.generate(
            prompt=prompt.strip(),
            negative_prompt=neg_prompt,
            video=None,
            mask=None,
            reference_images=refs if refs else None,
            conditioning_scale=float(cond_scale),
            height=h,
            width=w,
            num_frames=int(num_frames),
            num_inference_steps=int(steps),
            guidance_scale=float(cfg),
            flow_shift=float(shift),
            seed=int(seed),
        )
        final_mp4 = _save_to_drive(out_mp4, prefix='vace_r2v')
        progress(1.0, desc='Done!')
        return final_mp4, f'✅ **{mode_str}** completed in **{time.time() - t0:.1f}s** → `{final_mp4}`'
    except Exception as exc:
        traceback.print_exc()
        return None, f'❌ Error: {exc}'

def ui_run_v2v(
    variant, quantize_int8, offload, prompt, neg_prompt,
    src_video, mask_mode, mask_media, bbox_x1, bbox_y1, bbox_x2, bbox_y2, invert_mask, ref_img,
    res_label, num_frames, steps, cfg, cond_scale, shift, seed,
    progress=gr.Progress(),
):
    if not src_video:
        return None, '❌ Please upload a source video for V2V / Masked Inpainting.'
    if not prompt or not prompt.strip():
        return None, '❌ Please enter a prompt describing the target edit or scene.'
    try:
        progress(0.05, desc=f'Loading {variant} ...')
        ENGINE.load(variant=variant, offload=bool(offload), quantize_int8=bool(quantize_int8))
        h, w = _resolve_res(res_label)
        progress(0.12, desc='Preparing source video frames & binary conditioning mask ...')
        v_frames, m_frames = prepare_v2v_inpaint_video_and_mask(
            source_video_path=src_video,
            height=h,
            width=w,
            num_frames=int(num_frames),
            mask_mode=mask_mode,
            mask_media_path=mask_media,
            bbox_norm=(float(bbox_x1), float(bbox_y1), float(bbox_x2), float(bbox_y2)),
            invert_mask=bool(invert_mask),
        )
        refs = [ref_img] if ref_img is not None else None
        progress(0.20, desc=f'Denoising {len(v_frames)} frames at {w}x{h} ({int(steps)} steps) ...')
        t0 = time.time()
        out_mp4 = ENGINE.generate(
            prompt=prompt.strip(),
            negative_prompt=neg_prompt,
            video=v_frames,
            mask=m_frames,
            reference_images=refs,
            conditioning_scale=float(cond_scale),
            height=h,
            width=w,
            num_frames=len(v_frames),
            num_inference_steps=int(steps),
            guidance_scale=float(cfg),
            flow_shift=float(shift),
            seed=int(seed),
        )
        final_mp4 = _save_to_drive(out_mp4, prefix='vace_v2v')
        progress(1.0, desc='Done!')
        return final_mp4, f'✅ **V2V / Inpainting** completed in **{time.time() - t0:.1f}s** → `{final_mp4}`'
    except Exception as exc:
        traceback.print_exc()
        return None, f'❌ Error: {exc}'

def ui_run_flf_outpaint(
    variant, quantize_int8, offload, prompt, neg_prompt,
    cond_mode, first_img, last_img, outpaint_video,
    pad_top, pad_bottom, pad_left, pad_right, ref_img,
    res_label, num_frames, steps, cfg, cond_scale, shift, seed,
    progress=gr.Progress(),
):
    if not prompt or not prompt.strip():
        return None, '❌ Please enter a prompt.'
    try:
        progress(0.05, desc=f'Loading {variant} ...')
        ENGINE.load(variant=variant, offload=bool(offload), quantize_int8=bool(quantize_int8))
        h, w = _resolve_res(res_label)
        n_frames = snap_frames(int(num_frames))
        if cond_mode.startswith('First / Last'):
            if first_img is None:
                return None, '❌ Please upload at least a First Keyframe image.'
            v_frames, m_frames = prepare_flf2v_video_and_mask(
                first_img=first_img, last_img=last_img, height=h, width=w, num_frames=n_frames
            )
        else:
            if outpaint_video:
                src_frames, _ = read_video_frames_pil(outpaint_video, width=w, height=h, max_frames=n_frames)
            elif first_img is not None:
                src_frames = [to_rgb_pil(first_img, target_size=(w, h))] * n_frames
            else:
                return None, '❌ Please upload a First Keyframe image or a Source Video to outpaint.'
            v_frames, m_frames = prepare_outpaint_video_and_mask(
                source_frames=src_frames,
                height=h,
                width=w,
                num_frames=len(src_frames),
                pad_top=int(pad_top),
                pad_bottom=int(pad_bottom),
                pad_left=int(pad_left),
                pad_right=int(pad_right),
            )
        refs = [ref_img] if ref_img is not None else None
        progress(0.20, desc=f'Denoising {len(v_frames)} frames at {w}x{h} ({int(steps)} steps) ...')
        t0 = time.time()
        out_mp4 = ENGINE.generate(
            prompt=prompt.strip(),
            negative_prompt=neg_prompt,
            video=v_frames,
            mask=m_frames,
            reference_images=refs,
            conditioning_scale=float(cond_scale),
            height=h,
            width=w,
            num_frames=len(v_frames),
            num_inference_steps=int(steps),
            guidance_scale=float(cfg),
            flow_shift=float(shift),
            seed=int(seed),
        )
        final_mp4 = _save_to_drive(out_mp4, prefix='vace_flf_outpaint')
        progress(1.0, desc='Done!')
        return final_mp4, f'✅ **{cond_mode}** completed in **{time.time() - t0:.1f}s** → `{final_mp4}`'
    except Exception as exc:
        traceback.print_exc()
        return None, f'❌ Error: {exc}'

def get_vram_info():
    if not torch.cuda.is_available():
        return 'No CUDA GPU detected.'
    alloc = torch.cuda.memory_allocated() / (1024 ** 3)
    resv = torch.cuda.memory_reserved() / (1024 ** 3)
    tot = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    return (
        f'GPU: {torch.cuda.get_device_name(0)}\n'
        f'Allocated: {alloc:.2f} GB | Reserved: {resv:.2f} GB | Total: {tot:.2f} GB\n'
        f"Loaded Variant: {ENGINE.variant or 'None'} | INT8: {ENGINE.quantized} | Group Offload: {ENGINE.offloaded}"
    )

def free_vram():
    ENGINE.unload()
    return get_vram_info()

with gr.Blocks(title='Wan 2.1 VACE Studio', theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        '# 🎬 Wan 2.1 VACE — All-in-One Video Creation & Editing Studio\n'
        'Unified **Reference-to-Video (R2V)**, **Text-to-Video**, **Masked Video Inpainting (Swap/Move-Anything)**, '
        '**First/Last Keyframe (`FLF2V`)**, and **Spatial Video Outpainting** powered by `WanVACEPipeline`.'
    )

    with gr.Row():
        variant_dd = gr.Dropdown(
            choices=list(MODEL_REGISTRY.keys()),
            value=DEFAULT_VARIANT,
            label='Wan-VACE Model Variant',
            info='1.3B-VACE runs fast on T4 (16 GB) & L4 (24 GB). 14B-VACE is the flagship model for L4 (with INT8) & A100.',
        )
        res_dd = gr.Dropdown(
            choices=RES_NAMES,
            value=RES_NAMES[0],
            label='Target Resolution',
            info='480p (832x480 or 480x832) is recommended on T4/L4; 720p (1280x720) on L4/A100.',
        )
        quantize_cb = gr.Checkbox(
            value=False,
            label='Enable torchao INT8 Quantization',
            info='Auto-enabled for 14B-VACE on <32 GB GPUs; optional on 1.3B-VACE for extra VRAM headroom.',
        )
        offload_cb = gr.Checkbox(
            value=True,
            label='Enable Block-Level Group Offload',
            info='Streams transformer blocks between CPU and GPU with CUDA streams to fit 16–24 GB GPUs cleanly.',
        )

    with gr.Accordion('⚙️ Shared Sampling & Guidance Settings', open=False):
        with gr.Row():
            frames_sl = gr.Slider(
                17, 81, value=49, step=4, label='Number of Frames (4k + 1)',
                info='At 16 FPS: 33 frames ≈ 2.0s, 49 frames ≈ 3.0s, 81 frames ≈ 5.0s.',
            )
            steps_sl = gr.Slider(
                10, 50, value=30, step=1, label='Inference Steps',
                info='25–30 steps balances speed and quality; 40–50 steps for maximum detail.',
            )
            cfg_sl = gr.Slider(
                1.0, 10.0, value=5.0, step=0.5, label='Guidance Scale (CFG)',
                info='5.0 is the upstream recommended classifier-free guidance scale for Wan 2.1 VACE.',
            )
        with gr.Row():
            cond_scale_sl = gr.Slider(
                0.1, 2.0, value=1.0, step=0.05, label='VACE Conditioning Scale',
                info='Controls strength of the VACE control latent stream across all control blocks (default 1.0).',
            )
            shift_sl = gr.Slider(
                1.0, 10.0, value=3.0, step=0.5, label='UniPC Flow Shift',
                info='Use 3.0 for 480p generation and 5.0 for 720p generation.',
            )
            seed_sl = gr.Slider(
                -1, 2147483647, value=42, step=1, label='Random Seed (-1 for random)',
                info='Set a fixed integer seed for reproducible generations.',
            )
        neg_tb = gr.Textbox(
            value=NEGATIVE_PROMPT_DEFAULT, lines=2, label='Negative Prompt',
            info='Default negative prompt tuned for Wan 2.1 VACE temporal and visual quality.',
        )

    with gr.Tabs():
        with gr.Tab('1. 🖼️ Reference-to-Video (R2V) & Text-to-Video'):
            with gr.Row():
                with gr.Column(scale=3):
                    r2v_prompt = gr.Textbox(
                        label='Scene & Action Prompt',
                        value='A cinematic tracking shot of the character walking through a neon-lit cyberpunk night market in gentle rain, warm reflections on wet pavement, 4k film look.',
                        lines=3,
                        info='Describe how your uploaded reference subject(s) should move and interact in the scene.',
                    )
                    with gr.Row():
                        ref1_in = gr.Image(type='pil', label='Reference Image 1 (Optional Subject / Face)', height=200)
                        ref2_in = gr.Image(type='pil', label='Reference Image 2 (Optional Object / Outfit)', height=200)
                        ref3_in = gr.Image(type='pil', label='Reference Image 3 (Optional Background / Prop)', height=200)
                    r2v_btn = gr.Button('🎬 Generate Reference-to-Video / T2V', variant='primary')
                with gr.Column(scale=2):
                    r2v_video_out = gr.Video(label='Generated R2V / T2V Output', height=380)
                    r2v_status = gr.Markdown('Ready.')
            r2v_btn.click(
                ui_run_r2v,
                inputs=[
                    variant_dd, quantize_cb, offload_cb, r2v_prompt, neg_tb,
                    ref1_in, ref2_in, ref3_in, res_dd, frames_sl, steps_sl, cfg_sl, cond_scale_sl, shift_sl, seed_sl,
                ],
                outputs=[r2v_video_out, r2v_status],
            )

        with gr.Tab('2. 🎭 Video-to-Video & Masked Inpainting (Swap / Move-Anything)'):
            with gr.Row():
                with gr.Column(scale=3):
                    v2v_prompt = gr.Textbox(
                        label='Target Edit / Inpainting Prompt',
                        value='A futuristic silver robot performing the exact same motion in a sunlit studio, high detail, photorealistic.',
                        lines=3,
                        info='Describe what should appear inside the white masked region (or across the whole video).',
                    )
                    with gr.Row():
                        v2v_src_vid = gr.Video(label='Source Video (.mp4)', height=240)
                        v2v_ref_img = gr.Image(type='pil', label='Optional Reference Subject to Swap In', height=240)
                    v2v_mask_mode = gr.Dropdown(
                        choices=[
                            'Bounding Box Region (Inpaint / Swap)',
                            'Uploaded Mask Video / Image (White=Edit, Black=Keep)',
                            'Full Video Restyle (All Frames Editable)',
                        ],
                        value='Bounding Box Region (Inpaint / Swap)',
                        label='Masking Mode',
                        info='Choose how the editable region (white=255) vs preserved region (black=0) is constructed.',
                    )
                    v2v_mask_media = gr.File(
                        label='Optional Mask Video (.mp4) or Mask Image (.png/.jpg) — used when Masking Mode = Uploaded Mask',
                    )
                    with gr.Row():
                        bbox_x1 = gr.Slider(0.0, 0.9, value=0.20, step=0.05, label='Box Left (x1)', info='Normalized left edge (0.0–1.0) of the inpaint region.')
                        bbox_y1 = gr.Slider(0.0, 0.9, value=0.10, step=0.05, label='Box Top (y1)', info='Normalized top edge (0.0–1.0) of the inpaint region.')
                        bbox_x2 = gr.Slider(0.1, 1.0, value=0.80, step=0.05, label='Box Right (x2)', info='Normalized right edge (0.0–1.0) of the inpaint region.')
                        bbox_y2 = gr.Slider(0.1, 1.0, value=0.90, step=0.05, label='Box Bottom (y2)', info='Normalized bottom edge (0.0–1.0) of the inpaint region.')
                    v2v_invert = gr.Checkbox(
                        value=False, label='Invert Mask (edit outside box / invert uploaded mask)',
                        info='Flips black (0) and white (255) mask regions so the background is replaced while keeping the foreground.',
                    )
                    v2v_btn = gr.Button('🎭 Run V2V / Masked Inpainting', variant='primary')
                with gr.Column(scale=2):
                    v2v_video_out = gr.Video(label='Edited V2V / Inpainted Output', height=380)
                    v2v_status = gr.Markdown('Ready.')
            v2v_btn.click(
                ui_run_v2v,
                inputs=[
                    variant_dd, quantize_cb, offload_cb, v2v_prompt, neg_tb,
                    v2v_src_vid, v2v_mask_mode, v2v_mask_media, bbox_x1, bbox_y1, bbox_x2, bbox_y2, v2v_invert, v2v_ref_img,
                    res_dd, frames_sl, steps_sl, cfg_sl, cond_scale_sl, shift_sl, seed_sl,
                ],
                outputs=[v2v_video_out, v2v_status],
            )

        with gr.Tab('3. 🎞️ First/Last Keyframe (FLF2V) & Outpainting'):
            with gr.Row():
                with gr.Column(scale=3):
                    flf_mode = gr.Dropdown(
                        choices=[
                            'First / Last Frame Interpolation (FLF2V / I2V)',
                            'Spatial Video / Image Outpainting (Expand Canvas Margins)',
                        ],
                        value='First / Last Frame Interpolation (FLF2V / I2V)',
                        label='Conditioning Task',
                        info='FLF2V pins frame 0 (and optional final frame) with a black mask and generates intermediate frames. Outpainting expands borders around your image/video.',
                    )
                    flf_prompt = gr.Textbox(
                        label='Motion / Outpainting Prompt',
                        value='Smooth natural camera movement and realistic subject motion across the scene, cinematic lighting.',
                        lines=3,
                        info='Describe the transition between first and last keyframes, or the surrounding environment for outpainting.',
                    )
                    with gr.Row():
                        flf_first = gr.Image(type='pil', label='First Keyframe (Required for FLF2V / Image Outpaint)', height=220)
                        flf_last = gr.Image(type='pil', label='Last Keyframe (Optional — for FLF2V)', height=220)
                        flf_ref = gr.Image(type='pil', label='Optional Reference Subject Image', height=220)
                    flf_src_vid = gr.Video(label='Optional Source Video for Spatial Video Outpainting', height=200)
                    with gr.Row():
                        pad_t = gr.Slider(0, 160, value=64, step=16, label='Outpaint Top Margin (px)', info='Top border in pixels to synthesize in Outpainting mode.')
                        pad_b = gr.Slider(0, 160, value=64, step=16, label='Outpaint Bottom Margin (px)', info='Bottom border in pixels to synthesize in Outpainting mode.')
                        pad_l = gr.Slider(0, 256, value=128, step=16, label='Outpaint Left Margin (px)', info='Left border in pixels to synthesize in Outpainting mode.')
                        pad_r = gr.Slider(0, 256, value=128, step=16, label='Outpaint Right Margin (px)', info='Right border in pixels to synthesize in Outpainting mode.')
                    flf_btn = gr.Button('🎞️ Run Keyframe / Outpainting Generation', variant='primary')
                with gr.Column(scale=2):
                    flf_video_out = gr.Video(label='FLF2V / Outpainted Video Output', height=380)
                    flf_status = gr.Markdown('Ready.')
            flf_btn.click(
                ui_run_flf_outpaint,
                inputs=[
                    variant_dd, quantize_cb, offload_cb, flf_prompt, neg_tb,
                    flf_mode, flf_first, flf_last, flf_src_vid,
                    pad_t, pad_b, pad_l, pad_r, flf_ref,
                    res_dd, frames_sl, steps_sl, cfg_sl, cond_scale_sl, shift_sl, seed_sl,
                ],
                outputs=[flf_video_out, flf_status],
            )

        with gr.Tab('4. 🧠 VRAM & Engine Status'):
            vram_box = gr.Textbox(
                label='GPU & Pipeline Status', value=get_vram_info, lines=4, interactive=False,
                info='Displays current CUDA memory allocation and loaded Wan-VACE variant.',
            )
            with gr.Row():
                gr.Button('🔄 Refresh Status').click(get_vram_info, outputs=[vram_box])
                gr.Button('🧹 Unload Pipeline & Free VRAM').click(free_vram, outputs=[vram_box])

clear_output()
print('[UI] Launching Wan 2.1 VACE Gradio Studio ...')
demo.queue(max_size=4, concurrency_limit=1)
demo.load(lambda: '✅ **Wan 2.1 VACE Studio ready** — choose R2V/T2V, V2V Inpainting, or First/Last Keyframe & Outpainting.', outputs=[r2v_status])
demo.launch(share=True, show_error=True, height=880, prevent_thread_lock=False)

In [ ]:
# @title STEP 5 — Keep-Alive (anti-disconnect)
# @markdown Pings Hugging Face API every 60 seconds in a daemon thread to keep the Colab runtime active.

import time, threading, datetime, requests, IPython
_stop_keepalive = threading.Event()

def _pinger():
    while not _stop_keepalive.is_set():
        try:
            requests.get('https://huggingface.co/api/models/Wan-AI/Wan2.1-VACE-1.3B-diffusers', timeout=5)
            IPython.display.clear_output(wait=True)
            print(f"[Keep-Alive] {datetime.datetime.now().strftime('%H:%M:%S')} OK — stop with `_stop_keepalive.set()`.")
        except Exception as exc:
            print(f'[Keep-Alive] Warning: {exc}')
        _stop_keepalive.wait(60)

_ka_thread = threading.Thread(target=_pinger, daemon=True)
_ka_thread.start()
print('[Keep-Alive] Daemon started.')

In [ ]:
# @title STEP 6 — Quick Test (Headless Keyframe / R2V Smoke Test)
# @markdown Runs a fast end-to-end `WanVACEPipeline` generation without launching the Gradio UI and displays a download link.

import os, time, shutil, traceback
from PIL import Image, ImageDraw
from IPython.display import display, FileLink

QUICK_VARIANT = '1.3B-VACE (Fast, T4/L4)'  # @param ['1.3B-VACE (Fast, T4/L4)', '14B-VACE (Flagship, L4/A100)']
QUICK_PROMPT = 'A glowing paper lantern floating gently upward into a starry night sky over calm water, cinematic lighting.'  # @param {type:"string"}
QUICK_HEIGHT = 480  # @param {type:"integer"}
QUICK_WIDTH = 832  # @param {type:"integer"}
QUICK_FRAMES = 33  # @param {type:"integer"}
QUICK_STEPS = 20  # @param {type:"integer"}
QUICK_SEED = 42  # @param {type:"integer"}

try:
    t0 = time.time()
    ENGINE.load(variant=QUICK_VARIANT, offload=True, quantize_int8=QUICK_VARIANT.startswith('14B'))
    out_mp4 = ENGINE.generate(
        prompt=QUICK_PROMPT,
        height=QUICK_HEIGHT,
        width=QUICK_WIDTH,
        num_frames=QUICK_FRAMES,
        num_inference_steps=QUICK_STEPS,
        seed=QUICK_SEED,
    )
    final_path = DRIVE_OUT / f'quicktest_vace_{int(time.time())}.mp4'
    shutil.copy2(out_mp4, str(final_path))
    print(f'[QuickTest] Completed in {time.time() - t0:.1f}s -> {final_path}')
    display(FileLink(str(final_path)))
except Exception as exc:
    traceback.print_exc()
    raise

In [ ]:
# @title STEP 7 — Batch Video Generation from Prompt List
# @markdown Reads prompts from `BATCH_PROMPTS_FILE` (one prompt per line, with optional reference image) and exports `.mp4` clips to `DRIVE_OUT`.

import os, time, shutil, pathlib, traceback
from tqdm import tqdm
from PIL import Image
from IPython.display import display, FileLink

BATCH_VARIANT = '1.3B-VACE (Fast, T4/L4)'  # @param ['1.3B-VACE (Fast, T4/L4)', '14B-VACE (Flagship, L4/A100)']
BATCH_PROMPTS_FILE = '/content/wan_vace_batch_prompts.txt'  # @param {type:"string"}
BATCH_REF_IMAGE = ''  # @param {type:"string"}
BATCH_HEIGHT = 480  # @param {type:"integer"}
BATCH_WIDTH = 832  # @param {type:"integer"}
BATCH_FRAMES = 33  # @param {type:"integer"}
BATCH_STEPS = 25  # @param {type:"integer"}
BATCH_SEED = 42  # @param {type:"integer"}

prompts_path = pathlib.Path(BATCH_PROMPTS_FILE)
if not prompts_path.exists():
    default_prompts = [
        'A cinematic shot of a vintage train crossing a misty mountain viaduct at sunrise, 4k film look.',
        'A cute golden retriever puppy running across a sunlit meadow with wildflowers, slow motion.',
        'An astronaut walking through a bioluminescent alien forest at twilight, volumetric lighting.',
    ]
    prompts_path.write_text('\n'.join(default_prompts) + '\n', encoding='utf-8')
    print(f'[Batch] Created sample prompt list at {prompts_path}')

prompts = [l.strip() for l in prompts_path.read_text(encoding='utf-8').splitlines() if l.strip() and not l.strip().startswith('#')]
ref_list = [Image.open(BATCH_REF_IMAGE).convert('RGB')] if BATCH_REF_IMAGE.strip() and os.path.exists(BATCH_REF_IMAGE.strip()) else None

ENGINE.load(variant=BATCH_VARIANT, offload=True, quantize_int8=BATCH_VARIANT.startswith('14B'))
ok_count, fail_count = 0, 0
for idx, p_text in enumerate(tqdm(prompts, desc='Wan-VACE Batch'), 1):
    try:
        t0 = time.time()
        out_mp4 = ENGINE.generate(
            prompt=p_text,
            reference_images=ref_list,
            height=BATCH_HEIGHT,
            width=BATCH_WIDTH,
            num_frames=BATCH_FRAMES,
            num_inference_steps=BATCH_STEPS,
            seed=BATCH_SEED + idx,
        )
        slug = ''.join(c if c.isalnum() else '_' for c in p_text[:28]).strip('_') or f'clip_{idx:03d}'
        dest = DRIVE_OUT / f'batch_{idx:03d}_{slug}.mp4'
        shutil.copy2(out_mp4, str(dest))
        print(f'  [{idx:03d}/{len(prompts)}] OK ({time.time() - t0:.1f}s) -> {dest.name}')
        display(FileLink(str(dest)))
        ok_count += 1
    except Exception as exc:
        fail_count += 1
        print(f'  [{idx:03d}/{len(prompts)}] FAILED: {exc}')
        traceback.print_exc()

print(f'[Batch] Finished: {ok_count} succeeded, {fail_count} failed. Output folder: {DRIVE_OUT}')